In [4]:
import sqlite3
import pandas as pd
import sys
import os
from pathlib import Path
import warnings
from pathlib import Path
from darts import TimeSeries, concatenate
from darts.models import LinearRegressionModel

# Add the parent directory to sys.path so 'dashboard' can be imported if it's a local package
current_dir = Path(os.getcwd())
parent_dir = current_dir.parent
if str(parent_dir) not in sys.path:
	sys.path.insert(0, str(parent_dir))

from dashboard.database.db_connection import DatabaseConnection
# Get database path

db_path = parent_dir / "dashboard" / "database" / "bikes.db"
print(db_path)
# Initialize database connection
db = DatabaseConnection(str(db_path))

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")


INFO:dashboard.database.db_connection:Connected to database: /Users/ansen/Documents/GitHub/applied-data-science/dashboard/database/bikes.db


/Users/ansen/Documents/GitHub/applied-data-science/dashboard/database/bikes.db


In [5]:


# Model Paths (Adjust if necessary)
MODEL_24H_PATH = "../models/linear_24h_model.pkl"
MODEL_3D_PATH = "../models/linear_3d_model.pkl" 

Selected_Date_Start= "2018-11-01"
Selected_Date_END = "2018-11-30"
# ==========================================


In [7]:


def preprocess_data(df):
    """
    Transforms the database dataframe into the format required by the model.
    """
    print("Preprocessing data from database...")
    
    # 1. Create Datetime Column
    df['datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['hour'].astype(str) + ':00:00')
    
    # --- FIX START: Remove Duplicates ---
    # We check for duplicate timestamps and keep the first occurrence.
    initial_count = len(df)
    df = df.drop_duplicates(subset=['datetime'], keep='first')
    dropped_count = initial_count - len(df)
    if dropped_count > 0:
        print(f"Warning: Dropped {dropped_count} duplicate rows found in database.")
    # --- FIX END ---

    # 2. Set Index
    df.set_index('datetime', inplace=True)
    df = df.sort_index()

    # 3. Rename columns if necessary
    if 'rented_bike_count' in df.columns:
        df.rename(columns={'rented_bike_count': 'hourly_count'}, inplace=True)

    # 4. Feature Engineering
    numerical_features = [
        'hourly_count', 'temperature', 'humidity', 'wind_speed', 'visibility',
        'dew_point_temperature', 'solar_radiation', 'rainfall', 'snowfall'
    ]
    
    # Handle potential column name mismatches (e.g. typos in source data)
    available_cols = df.columns.tolist()
    final_numerical = [c for c in numerical_features if c in available_cols]
    
    categorical_cols = ['seasons', 'holiday', 'functioning_day']
   
    
    # Create Dummies
    df_categorical = df[categorical_cols]
    df_dummies = pd.get_dummies(df_categorical)
    df_dummies = df_dummies.replace({False: 0, True: 1})
    
    # Merge
    df_processed = pd.merge(df[final_numerical], df_dummies, left_index=True, right_index=True)
    
    return df_processed


In [ ]:
# 2. Load Data from DB

"""Fetches all data from the staging table."""
query = "SELECT * FROM stg_bike_rentals_hourly ORDER BY date, hour"
df_raw = pd.read_sql_query(query, db.connection)

# 3. Preprocess Data
df_processed = preprocess_data(df_raw)

Preprocessing data from database...


In [11]:

print(df_processed.tail())

                     hourly_count  temperature  humidity  wind_speed  \
datetime                                                               
2018-11-30 19:00:00          1003          4.2        34         2.6   
2018-11-30 20:00:00           764          3.4        37         2.3   
2018-11-30 21:00:00           694          2.6        39         0.3   
2018-11-30 22:00:00           712          2.1        41         1.0   
2018-11-30 23:00:00           584          1.9        43         1.3   

                     visibility  dew_point_temperature  solar_radiation  \
datetime                                                                  
2018-11-30 19:00:00        1894                  -10.3              0.0   
2018-11-30 20:00:00        2000                   -9.9              0.0   
2018-11-30 21:00:00        1968                   -9.9              0.0   
2018-11-30 22:00:00        1859                   -9.8              0.0   
2018-11-30 23:00:00        1909              

In [12]:
MODEL_24H_PATH = "../models/linear_24h_model.pkl"
model = LinearRegressionModel.load(MODEL_24H_PATH)
ts_series = TimeSeries.from_dataframe(df_processed[['hourly_count']], freq='h')

features_df = df_processed.drop(['hourly_count'], axis=1)
features_series = TimeSeries.from_dataframe(features_df, freq='h')


In [ ]:
print(pd.Timestamp(begin_date))

2018-11-01 00:00:00


In [72]:
horizon_hours = 24
begin_date = "2018-11-01"
# histoical_forecasts simulates 'time-travel' to predict future points
preds = model.historical_forecasts(
    series=ts_series,
    future_covariates=features_series,
    forecast_horizon=1,
    start=pd.Timestamp(begin_date),
    stride=1,
    last_points_only=True,
    retrain=False, # Use pre-trained weights
)

In [70]:
preds

,hourly_count
datetime,
2017-12-02 00:00:00,162.998775
2017-12-02 01:00:00,208.470979
2017-12-02 02:00:00,186.618580
2017-12-02 03:00:00,94.971066
2017-12-02 04:00:00,106.634704
...,...
2018-11-30 19:00:00,1133.982726
2018-11-30 20:00:00,836.496020
2018-11-30 21:00:00,786.210290


In [53]:
def generate_predictions(df, model_path, horizon_hours, begin_datetime):
    """Loads a model and generates historical forecasts."""
    print(f"\n--- Processing {horizon_hours}-hour Prediction ---")
    
    if not Path(model_path).exists():
        print(f"Error: Model file not found at {model_path}")
        return None

    print(f"Loading model from {model_path}...")
    try:
        model = LinearRegressionModel.load(model_path)
    except Exception as e:
        print(f"Failed to load model: {e}")
        return None

    print("Preparing TimeSeries data...")
    # Target series
    ts_series = TimeSeries.from_dataframe(df[['hourly_count']], freq='h')
    
    # Covariates series (all other columns)
    features_df = df.drop(['hourly_count'], axis=1)
    features_series = TimeSeries.from_dataframe(features_df, freq='h')

    print(f"Generating forecasts at {begin_date} ...")
    
    try:
        # historical_forecasts simulates 'time-travel' to predict future points
        preds = model.historical_forecasts(
            series=ts_series,
            future_covariates=features_series,
            forecast_horizon=horizon_hours,
            stride=1,
            start=pd.Timestamp(begin_datetime),
            last_points_only=True,
            retrain=False, # Use pre-trained weights
        )
    
        return preds
    except Exception as e:
        print(f"Prediction failed: {e}")
        return None

In [54]:
begin_time_str = "2018-11-21 00:00:00"
preds = generate_predictions(df_processed, MODEL_24H_PATH, 24, begin_datetime=begin_time_str)


--- Processing 24-hour Prediction ---
Loading model from ../models/linear_24h_model.pkl...
Preparing TimeSeries data...
Generating forecasts at 2018-11-01 ...


In [57]:
print(preds)


                     hourly_count
datetime                         
2018-11-01 23:00:00    624.095847
2018-11-02 00:00:00    471.815251
2018-11-02 01:00:00    365.529428
2018-11-02 02:00:00    274.561867
2018-11-02 03:00:00    180.120533
...                           ...
2018-11-30 19:00:00   1054.310974
2018-11-30 20:00:00    879.583279
2018-11-30 21:00:00    812.675840
2018-11-30 22:00:00    849.192320
2018-11-30 23:00:00    622.814703

shape: (697, 1, 1), freq: h, size: 5.45 KB
